# Part 1 — Data Preprocessing & Cleaning

**Project:** Social Media and Stress Patterns
**Goal:** turn the raw CSV into a clean, trustworthy dataset that every other part can build on.

This notebook runs the complete cleaning pipeline from `scripts/preprocess.py`.
The pipeline performs **12 steps**, each printed in plain English:

| Step | What it does |
|------|--------------|
| 1 | Load the raw file |
| 2 | Profile the data (types, missing, duplicates, stats) |
| 3 | Rename columns to a clean naming style |
| 4 | Remove stray spaces in text columns |
| 5b | Report **which rows** have missing values (cell by cell) |
| 5 | Missing-value policy (median / "Unknown") — applied only if needed |
| 6 | Duplicate-row policy (drop exact copies) |
| 7 | Validate IDs (U001–U500, unique) |
| 8 | Range checks (1–10 scales, 0–24 h) |
| 9 | Outlier scan (IQR rule — reported, not deleted) |
| 10 | Fix column types (7.0 → 7) |
| 11 | Verify the derived screen-time group column (0 mismatches) |
| 12 | Final safety checks + save |

**Outputs:**
- `data/processed/social_media_stress_clean.csv` — the cleaned dataset (500 rows × 11 columns, 0 missing, 0 duplicates)
- `reports/data_cleaning_report.md` — the full record of checks, changes, and anomalies

In [1]:
# Locate the project root (works whether this notebook runs from the
# project folder or from inside notebooks/).
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "scripts" / "preprocess.py").exists():
    root = root.parent

script = root / "scripts" / "preprocess.py"
print("Project root:", root)

# Run the complete cleaning pipeline (same code as scripts/preprocess.py).
# Every step prints what it does and what it found.
%run $script

Project root: /Users/jazmingill/social-media-stress-analysis


STARTING THE CLEANING PIPELINE
STEP 1 - Loaded 500 rows and 11 columns from the raw file.

STEP 2 - DATA PROFILE (before cleaning)
Column types:
User_ID                          str
Age                            int64
Gender                           str
Daily_Screen_Time(hrs)       float64
Sleep_Quality(1-10)          float64
Stress_Level(1-10)           float64
Days_Without_Social_Media    float64
Exercise_Frequency(week)     float64
Social_Media_Platform            str
Happiness_Index(1-10)        float64
Screen_Time_Group                str

Missing values per column:
User_ID                      0
Age                          0
Gender                       0
Daily_Screen_Time(hrs)       0
Sleep_Quality(1-10)          0
Stress_Level(1-10)           0
Days_Without_Social_Media    0
Exercise_Frequency(week)     0
Social_Media_Platform        0
Happiness_Index(1-10)        0
Screen_Time_Group            0

Duplicate rows: 0

Unique values per column:
User_ID                      500


In [2]:
# Check the result: load the cleaned dataset and inspect it.
import pandas as pd

df = pd.read_csv(root / "data" / "processed" / "social_media_stress_clean.csv")

print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print()
print("Column types:")
print(df.dtypes)
print()
df.head(8)

Shape: (500, 11)
Missing values: 0
Duplicate rows: 0

Column types:
user_id                            str
age                              int64
gender                             str
daily_screen_time_hours        float64
sleep_quality                    int64
stress_level                     int64
days_without_social_media        int64
exercise_frequency_per_week      int64
social_media_platform              str
happiness_index                  int64
screen_time_group                  str
dtype: object



,user_id,age,gender,daily_screen_time_hours,sleep_quality,stress_level,days_without_social_media,exercise_frequency_per_week,social_media_platform,happiness_index,screen_time_group
0,U001,44,Male,3.1,7,6,2,5,Facebook,10,3-6 hrs
1,U002,30,Other,5.1,7,8,5,3,LinkedIn,10,3-6 hrs
2,U003,23,Other,7.4,6,7,1,3,YouTube,6,6-9 hrs
3,U004,36,Female,5.7,7,8,1,1,TikTok,8,3-6 hrs
4,U005,34,Female,7.0,4,7,5,1,X (Twitter),8,6-9 hrs
5,U006,38,Male,6.6,5,7,4,3,LinkedIn,8,6-9 hrs
6,U007,26,Female,7.8,4,8,2,0,TikTok,7,6-9 hrs
7,U008,26,Female,7.4,5,6,1,4,Instagram,7,6-9 hrs


**Cleaning result:**
- All 500 rows preserved (nothing dropped)
- 0 missing values · 0 duplicate rows
- Integer columns fixed (sleep, stress, exercise, happiness, offline days)
- One anomaly documented and retained: a single `days_without_social_media = 9` (exceeds a 7-day week; the reference period is not documented, so the value was kept and flagged rather than silently changed)
- Outliers reported (IQR rule) but retained — the flagged values are plausible real answers